This notebook allows you to run ACOLITE processing on Google Colab. The required dependencies are installed and the latest version is cloned from GitHub (https://github.com/acolite/acolite).

In this example, a NewSat image is retrieved from the Satellogic Sample Data public S3 bucket (see https://developers.satellogic.com/data/data-samples.html), processed and stored on in the Colab runtime environment. Ancillary data can be retrieved if EarthData credentials are set up, but are not essential for this example.

Author: Quinten Vanhellemont, RBINS, quinten.vanhellemont@naturalsciences.be

The next cell sets up the ACOLITE dependencies in Colab if they are not already there. This is specific to Colab and you will need to set up an ACOLITE environment on your machine if you want to run it locally. Here we add s3fs to transfer data from the S3 bucket.

In [ ]:
# general imports
import os, sys

# install dependencies for acolite
!pip install numpy matplotlib scipy gdal pyproj scikit-image pyhdf pyresample netcdf4 h5py requests pygrib cartopy s3fs

The ACOLITE code is cloned from the GitHub repository.

In [ ]:
# clone acolite repo if it does not exist
if not os.path.exists('acolite'):
  !git clone --depth 1 https://github.com/acolite/acolite

The ACOLITE code is imported as "ac" so the Python functions can be called directly.

In [ ]:
# add acolite to path and import
sys.path.append('acolite')
import acolite as ac

The next cell downloads a sample NewSat scene kindly provided by Satellogic.

In [ ]:
import s3fs
fs = s3fs.S3FileSystem(anon = True)
s3_files = fs.glob('s3://satellogic-sample-data/L1C/Baotou/18f16f7a-35c2-45ac-a7ec-d133df3b12b0/2_2_0/20240515_063346_504_SN43_L1C_MS_203086/20240515_063349_676_SN43_L1C_MS_*')

input_dir = 'Input/{}'.format(os.path.basename(os.path.dirname(s3_files[0])))
if not os.path.exists(input_dir): os.makedirs(input_dir)

for i, file in enumerate(s3_files):
  bn = os.path.basename(file)
  local_file = '{}/{}'.format(input_dir, bn)
  if not os.path.exists(local_file):
      print('Downloading {}'.format(bn))
      fs.download(file, local_file)


In [ ]:
## location of interest
## here for centering 0.5x0.5 km box
region_name = 'Baotou'
station_lon = 109.6291
station_lat = 40.8514
station_box_size = 0.5

The next cell sets up a Python dictionary with typical settings for processing ACOLITE/DSF for a region of interest (here a square box centred on the location from the cell above). DSF processing is based on Vanhellemont (2023) https://doi.org/10.1364/OE.483418 and references therein.

In [ ]:
# acolite settings
settings = {## basic input and output settings
            ## local_scenes list is used as returned from the API download function
            'inputfile': input_dir,
            'output': 'Output/DSF/',

            ## set up region of interest centred on lon/lat, with box size in km
            'region_name': region_name,
            'station_lon': station_lon,
            'station_lat': station_lat,
            'station_box_size': station_box_size,
            'station_box_units': 'km',

            ## disable ancillary data retrieval, set to True if EarthData credentials are set
            'ancillary_data': False,

            ## use DSF processing
            'atmospheric_correction_method': 'dark_spectrum',

            ## supply aerosol information
            'dsf_fixed_aot': 0.192,
            'dsf_fixed_lut': 'ACOLITE-LUT-202110-MOD1',

            ## change upper reflectance limit of RGBs
            'rgb_max': [0.25] * 3,

            ## delete L1R output NetCDF
            'l1r_delete_netcdf': True,
            ## don't output L1R RGBs
            'l1r_rgb_keys': [],
            ## delete ACOLITE text outputs
            'delete_acolite_run_text_files': True,
            }

The next cell passes the settings dictionary to ACOLITE and runs the processing. Note that the first run will be slightly slower, as the LUTs for the current sensor will have to be downloaded. A projection of the original data is automatically performed using Copernicus 30m DEM.

In [ ]:
## run acolite with settings dict
ret = ac.acolite.acolite_run(settings)

In [ ]:
## imports for plotting
import glob
import matplotlib.pyplot as plt
import numpy as np

Let's make a surface reflectance RGB!

In [ ]:
## create rhos RGB from DSF data
ncf = glob.glob('Output/DSF/*L2R*.nc')[0]

gatts = ac.shared.nc_gatts(ncf)
title = '{} {}'.format(gatts['sensor'].replace('_', '/'), gatts['isodate'][0:19])

lat = ac.shared.nc_data(ncf, 'lat')
lon = ac.shared.nc_data(ncf, 'lon')
datasets = ac.shared.nc_datasets(ncf)

stack = []
waves = []
for ds in datasets:
  if ds.startswith('rhos_'):
      waves.append(float(ds[ds.find('_')+1:]))
      stack.append(ac.shared.nc_data(ncf, ds))
stack = np.asarray(stack)
waves = np.asarray(waves)
rgb = ac.map.rgb.stack(waves, stack,  rgb_max = [0.25] * 3,)

fig, ax = plt.subplots()
plt.axis('off')
plt.title(title)

mesh = plt.imshow(rgb)
plt.savefig('Output/DSF/RGB.png', dpi = 300, bbox_inches = 'tight', facecolor = 'white')

Here we provide the approximate locations of the centres of the squares in the target.

In [ ]:
stations = {}
stations['1'] = 109.62867736816406, 40.85199737548828
stations['2'] = 109.6292495727539, 40.85196304321289
stations['3'] = 109.6286392211914, 40.851558685302734
stations['4'] =	109.62921905517578,	40.85152053833008

Let's make another RGB showing the selected positions.

In [ ]:
fig, ax = plt.subplots()
plt.axis('off')
plt.title(title)

mesh = plt.pcolormesh(lon, lat, rgb)
for st in stations:
  l = plt.plot(stations[st][0], stations[st][1], '.')
  plt.text(stations[st][0] + 0.00005, stations[st][1] - 0.00005, st, color = l[0].get_color())

plt.savefig('Output/DSF/RGB_points.png', dpi = 300, bbox_inches = 'tight', facecolor = 'white')

Now we extract the data for the four configured stations.

In [ ]:
extracts = {}
for st in stations:
  extracts[st] = ac.shared.nc_extract_point(ncf, stations[st][0], stations[st][1], box_size = 11)

And make plots of top-of-atmosphere (left) and surface (right) reflectance.

In [ ]:
fig, ax = plt.subplots(1,2, sharey = True, sharex = True)
for st in extracts:
  ext_dsf = extracts[st]

  plt.sca(ax[0])
  mean = np.asarray([np.nanmean(ext_dsf['data'][ds]) for ds in ext_dsf['rhot_datasets']])
  std = np.asarray([np.nanstd(ext_dsf['data'][ds]) for ds in ext_dsf['rhot_datasets']])
  plt.errorbar(ext_dsf['rhot_wave'], mean, marker = '.', linestyle = '--', yerr = std, label = st)
  plt.xlabel('Wavelength (nm)')
  plt.ylabel(r'$\rho$ (1)')
  plt.title(r'$\rho_t$')

  plt.sca(ax[1])
  mean = np.asarray([np.nanmean(ext_dsf['data'][ds]) for ds in ext_dsf['rhos_datasets']])
  std = np.asarray([np.nanstd(ext_dsf['data'][ds]) for ds in ext_dsf['rhos_datasets']])
  plt.errorbar(ext_dsf['rhos_wave'], mean, marker = '.', linestyle = '--', yerr = std, label = st)
  plt.xlabel('Wavelength (nm)')
  plt.title(r'$\rho_s$')

plt.sca(ax[0])
plt.ylim(0, 0.45)
plt.legend()